# Local Microphone Denoising Demo


In [ ]:
# Optional dependency install. Run once if your environment is missing packages.

import sys
import subprocess

INSTALL_DEPS = False

if INSTALL_DEPS:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch", "torchaudio", "soundfile", "scipy", "numpy", "sounddevice", "ipython"
    ])
    print("Installed dependencies.")
else:
    print("INSTALL_DEPS=False; assuming dependencies are already installed.")

In [ ]:
# Imports and user settings

from __future__ import annotations

import json
import math
import shutil
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Tuple

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import torchaudio
    TORCHAUDIO_AVAILABLE = True
except Exception as exc:
    torchaudio = None
    TORCHAUDIO_AVAILABLE = False
    print("torchaudio unavailable; using soundfile/scipy fallback:", repr(exc))

try:
    import sounddevice as sd
    SOUNDDEVICE_AVAILABLE = True
except Exception as exc:
    sd = None
    SOUNDDEVICE_AVAILABLE = False
    print("sounddevice unavailable; mic recording disabled:", repr(exc))

from IPython.display import Audio as IPAudio, display, Markdown

# ---- EDIT THESE ----
MODEL_PATH = Path("./general_audio_resunet_full_vb_gated_tcn_export.zip")  # .zip export or direct .pt checkpoint
OUTPUT_ROOT = Path("./demo_outputs/minimal_mic_demo")

RECORD_SECONDS = 8.0
MIC_SAMPLE_RATE = 48_000       
MODEL_STRENGTH = 0.85          # try 0.75–0.90 if output sounds overprocessed; 1.0 = full model
HOP_SECONDS = 1.0              # 50% overlap for 2-second chunks

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("MODEL_PATH:", MODEL_PATH.resolve())
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())
print("SOUNDDEVICE_AVAILABLE:", SOUNDDEVICE_AVAILABLE)

## Model definition

In [ ]:
def _num_groups(channels: int, max_groups: int = 8) -> int:
    for groups in range(min(max_groups, channels), 0, -1):
        if channels % groups == 0:
            return groups
    return 1


class ResBlock1d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 5):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=pad)
        self.norm1 = nn.GroupNorm(_num_groups(out_channels), out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=pad)
        self.norm2 = nn.GroupNorm(_num_groups(out_channels), out_channels)
        self.act = nn.SiLU()
        self.skip = nn.Identity() if in_channels == out_channels else nn.Conv1d(in_channels, out_channels, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.skip(x)
        y = self.act(self.norm1(self.conv1(x)))
        y = self.norm2(self.conv2(y))
        return self.act(y + residual)


class TCNBlock1d(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 5, dilation: int = 1, dropout: float = 0.0):
        super().__init__()
        pad = dilation * (kernel_size - 1) // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.norm1 = nn.GroupNorm(_num_groups(channels), channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.norm2 = nn.GroupNorm(_num_groups(channels), channels)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        y = self.act(self.norm1(self.conv1(x)))
        y = self.dropout(y)
        y = self.norm2(self.conv2(y))
        return self.act(y + residual)


def match_length(x: torch.Tensor, target_length: int) -> torch.Tensor:
    current = x.shape[-1]
    if current == target_length:
        return x
    if current > target_length:
        return x[..., :target_length]
    return F.pad(x, (0, target_length - current))


class GatedTCNWaveformResUNet(nn.Module):
    def __init__(
        self,
        in_channels: int = 1,
        out_channels: int = 1,
        base_channels: int = 24,
        depth: int = 3,
        residual_scale: float = 0.5,
        gate_bias_init: float = -0.5,
        use_tcn_bottleneck: bool = True,
        tcn_blocks: int = 6,
        tcn_kernel_size: int = 5,
    ):
        super().__init__()
        self.depth = depth
        self.residual_scale = residual_scale

        self.enc_blocks = nn.ModuleList()
        self.downs = nn.ModuleList()
        channels = []
        ch = in_channels

        for level in range(depth):
            out_ch = base_channels * (2 ** level)
            self.enc_blocks.append(ResBlock1d(ch, out_ch))
            self.downs.append(nn.Conv1d(out_ch, out_ch, kernel_size=4, stride=2, padding=1))
            channels.append(out_ch)
            ch = out_ch

        self.bottleneck = ResBlock1d(ch, ch * 2)
        ch = ch * 2

        if use_tcn_bottleneck and tcn_blocks > 0:
            dilations = [2 ** (i % 6) for i in range(tcn_blocks)]
            self.tcn = nn.Sequential(*[
                TCNBlock1d(ch, kernel_size=tcn_kernel_size, dilation=d)
                for d in dilations
            ])
        else:
            self.tcn = nn.Identity()

        self.ups = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        for skip_ch in reversed(channels):
            self.ups.append(nn.ConvTranspose1d(ch, skip_ch, kernel_size=4, stride=2, padding=1))
            self.dec_blocks.append(ResBlock1d(skip_ch * 2, skip_ch))
            ch = skip_ch

        self.residual_head = nn.Conv1d(ch, out_channels, kernel_size=1)
        self.gate_head = nn.Conv1d(ch, out_channels, kernel_size=1)

        nn.init.zeros_(self.residual_head.weight)
        nn.init.zeros_(self.residual_head.bias)
        nn.init.zeros_(self.gate_head.weight)
        nn.init.constant_(self.gate_head.bias, gate_bias_init)

    def forward(self, mixture: torch.Tensor, return_aux: bool = False):
        original = mixture
        original_length = mixture.shape[-1]
        multiple = 2 ** self.depth
        pad = (multiple - original_length % multiple) % multiple
        x = F.pad(mixture, (0, pad)) if pad else mixture

        skips = []
        for enc, down in zip(self.enc_blocks, self.downs):
            x = enc(x)
            skips.append(x)
            x = down(x)

        x = self.bottleneck(x)
        x = self.tcn(x)

        for up, dec, skip in zip(self.ups, self.dec_blocks, reversed(skips)):
            x = up(x)
            x = match_length(x, skip.shape[-1])
            x = torch.cat([x, skip], dim=1)
            x = dec(x)

        residual = match_length(self.residual_head(x), original_length)
        gate_logits = match_length(self.gate_head(x), original_length)
        gate = torch.sigmoid(gate_logits)
        pred = original[..., :original_length] + self.residual_scale * gate * residual

        if return_aux:
            edit = pred - original[..., :original_length]
            return pred, {
                "residual": residual,
                "gate": gate,
                "gate_mean": gate.mean(),
                "residual_rms": residual.pow(2).mean().sqrt(),
                "edit_rms": edit.pow(2).mean().sqrt(),
            }
        return pred

## Load checkpoint

In [ ]:
# Simple checkpoint loader.
# Assumes MODEL_PATH points to a zip containing best_si_sdri.pt.

from pathlib import Path
import shutil
import torch

MODEL_PATH = Path("./general_audio_resunet_full_vb_gated_tcn_export.zip")
EXTRACT_DIR = Path("./loaded_model")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Extract checkpoint zip.
shutil.unpack_archive(str(MODEL_PATH), str(EXTRACT_DIR))

# Locate best checkpoint.
ckpt_path = EXTRACT_DIR / "best_si_sdri.pt"

if not ckpt_path.exists():
    matches = list(EXTRACT_DIR.rglob("best_si_sdri.pt"))
    if not matches:
        raise FileNotFoundError(f"Could not find best_si_sdri.pt inside {MODEL_PATH}")
    ckpt_path = matches[0]

print("Loading checkpoint:", ckpt_path)

# Load checkpoint.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
saved_cfg = checkpoint["cfg"]

# Build model from saved config.
model = GatedTCNWaveformResUNet(
    in_channels=1,
    out_channels=1,
    base_channels=saved_cfg.get("base_channels", 24),
    depth=saved_cfg.get("depth", 3),
    residual_scale=saved_cfg.get("residual_scale", 0.5),
    gate_bias_init=saved_cfg.get("gate_bias_init", -0.5),
    use_tcn_bottleneck=saved_cfg.get("use_tcn_bottleneck", True),
    tcn_blocks=saved_cfg.get("tcn_blocks", 6),
    tcn_kernel_size=saved_cfg.get("tcn_kernel_size", 5),
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully.")
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print("Device:", device)

## Live microphone demo

In [ ]:
def to_mono_np(audio: np.ndarray) -> np.ndarray:
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    return audio.astype(np.float32)


def resample_np(audio: np.ndarray, orig_sr: int, target_sr: int) -> np.ndarray:
    if orig_sr == target_sr:
        return audio.astype(np.float32)
    g = math.gcd(int(orig_sr), int(target_sr))
    up, down = target_sr // g, orig_sr // g
    return resample_poly(audio, up=up, down=down).astype(np.float32)


def write_wav(path: str | Path, audio: np.ndarray, sr: int):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    audio = np.asarray(audio, dtype=np.float32)
    audio = np.nan_to_num(audio)
    audio = np.clip(audio, -1.0, 1.0)
    sf.write(str(path), audio, sr)


def peak_normalize_np(audio: np.ndarray, eps: float = 1e-8):
    scale = max(float(np.max(np.abs(audio))), eps)
    return (audio / scale).astype(np.float32), scale


def rms_db(audio, eps=1e-8):
    audio = np.asarray(audio, dtype=np.float32)
    rms = np.sqrt(np.mean(audio ** 2) + eps)
    return float(20.0 * np.log10(rms + eps))


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if torch.is_tensor(obj):
        if obj.numel() == 1:
            return float(obj.detach().cpu())
        return obj.detach().cpu().tolist()
    if isinstance(obj, Path):
        return str(obj)
    return obj


def spectral_gate_baseline_np(x_norm: np.ndarray, sr: int, n_fft: int = 1024, hop: int = 256,
                              noise_quantile: float = 0.20, strength: float = 0.85,
                              floor: float = 0.08) -> np.ndarray:
    # Simple spectral gate baseline. Input/output are normalized mono float arrays.
    x = torch.from_numpy(x_norm.astype(np.float32)).view(1, -1)
    window = torch.hann_window(n_fft)
    spec = torch.stft(x, n_fft=n_fft, hop_length=hop, win_length=n_fft, window=window, return_complex=True)
    mag = spec.abs()[0]
    phase = torch.angle(spec[0])

    noise_floor = torch.quantile(mag, q=noise_quantile, dim=1, keepdim=True)
    mask = 1.0 - strength * (noise_floor / (mag + 1e-8))
    mask = mask.clamp(min=floor, max=1.0)

    mask4 = mask.unsqueeze(0).unsqueeze(0)
    mask4 = F.avg_pool2d(F.pad(mask4, (1, 1, 1, 1), mode="replicate"), kernel_size=3, stride=1)
    mask = mask4[0, 0]

    enhanced = mag * mask * torch.exp(1j * phase)
    y = torch.istft(enhanced.unsqueeze(0), n_fft=n_fft, hop_length=hop, win_length=n_fft, window=window, length=x.shape[-1])
    return y[0].numpy().astype(np.float32)


@torch.no_grad()
def run_model_chunked_norm(model: nn.Module, x_norm: torch.Tensor, chunk_samples: int, hop_samples: int) -> torch.Tensor:
    # x_norm: [T] normalized waveform tensor. Returns [T].
    model.eval()
    device = next(model.parameters()).device
    x_norm = x_norm.float().to(device)
    length = x_norm.numel()

    if length == 0:
        return x_norm.cpu()

    if length <= chunk_samples:
        chunk = F.pad(x_norm, (0, chunk_samples - length)).view(1, 1, -1)
        pred = model(chunk)
        if isinstance(pred, tuple):
            pred = pred[0]
        return pred[0, 0, :length].detach().cpu()

    window = torch.hann_window(chunk_samples, device=device).clamp_min(1e-3)
    out = torch.zeros(length + chunk_samples, device=device)
    weight = torch.zeros(length + chunk_samples, device=device)

    starts = list(range(0, max(1, length - chunk_samples + 1), hop_samples))
    if starts[-1] != length - chunk_samples:
        starts.append(max(0, length - chunk_samples))

    for start in starts:
        chunk = x_norm[start:start + chunk_samples]
        if chunk.numel() < chunk_samples:
            chunk = F.pad(chunk, (0, chunk_samples - chunk.numel()))
        pred = model(chunk.view(1, 1, -1))
        if isinstance(pred, tuple):
            pred = pred[0]
        pred = pred[0, 0]
        out[start:start + chunk_samples] += pred * window
        weight[start:start + chunk_samples] += window

    y = out[:length] / weight[:length].clamp_min(1e-8)
    return y.detach().cpu()


def denoise_np(audio: np.ndarray, sr: int, model_strength: float = MODEL_STRENGTH) -> Dict[str, Any]:
    # Returns raw 16k, baseline, model output, sample rate, and timing.
    raw = to_mono_np(audio)
    raw_16k = resample_np(raw, sr, MODEL_SAMPLE_RATE)
    raw_16k = raw_16k - float(np.mean(raw_16k))  # remove DC offset
    x_norm, scale = peak_normalize_np(raw_16k)

    t0 = time.time()
    x_tensor = torch.from_numpy(x_norm)
    model_norm = run_model_chunked_norm(model, x_tensor, MODEL_CHUNK_SAMPLES, HOP_SAMPLES).numpy()
    model_norm = x_norm + model_strength * (model_norm - x_norm)
    model_out = (model_norm * scale).astype(np.float32)
    elapsed = time.time() - t0

    baseline_norm = spectral_gate_baseline_np(x_norm, MODEL_SAMPLE_RATE)
    baseline_out = (baseline_norm * scale).astype(np.float32)

    duration = len(raw_16k) / MODEL_SAMPLE_RATE
    return {
        "raw_16k": raw_16k.astype(np.float32),
        "baseline": baseline_out,
        "model": model_out,
        "sample_rate": MODEL_SAMPLE_RATE,
        "duration_sec": duration,
        "model_elapsed_sec": elapsed,
        "model_rtf": elapsed / max(duration, 1e-8),
        "scale": scale,
        "model_strength": model_strength,
    }


def record_from_mic(seconds: float = RECORD_SECONDS, sr: int = MIC_SAMPLE_RATE) -> np.ndarray:
    if not SOUNDDEVICE_AVAILABLE:
        raise RuntimeError("sounddevice is not available. Install it with `pip install sounddevice` and enable microphone permissions.")
    print(f"Recording {seconds:.1f} seconds at {sr} Hz. Speak now...")
    audio = sd.rec(int(seconds * sr), samplerate=sr, channels=1, dtype="float32")
    sd.wait()
    print("Recording complete.")
    return audio[:, 0].astype(np.float32)


def mic_demo(seconds: float = RECORD_SECONDS, model_strength: float = MODEL_STRENGTH, name: str = "mic_demo"):
    run_dir = OUTPUT_ROOT / f"{name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    run_dir.mkdir(parents=True, exist_ok=True)

    raw_mic = record_from_mic(seconds=seconds, sr=MIC_SAMPLE_RATE)
    result = denoise_np(raw_mic, MIC_SAMPLE_RATE, model_strength=model_strength)

    raw_path = run_dir / "01_original_mic_16k.wav"
    base_path = run_dir / "02_spectral_gate_baseline.wav"
    model_path = run_dir / "03_model_prediction.wav"
    meta_path = run_dir / "metadata.json"

    write_wav(raw_path, result["raw_16k"], result["sample_rate"])
    write_wav(base_path, result["baseline"], result["sample_rate"])
    write_wav(model_path, result["model"], result["sample_rate"])

    meta = {
        "sample_rate": result["sample_rate"],
        "duration_sec": result["duration_sec"],
        "model_elapsed_sec": result["model_elapsed_sec"],
        "model_rtf": result["model_rtf"],
        "model_strength": result["model_strength"],
        "input_peak": float(np.max(np.abs(result["raw_16k"]))),
        "input_rms_db": rms_db(result["raw_16k"]),
        "baseline_rms_db": rms_db(result["baseline"]),
        "model_rms_db": rms_db(result["model"]),
        "raw_path": str(raw_path),
        "baseline_path": str(base_path),
        "model_path": str(model_path),
    }
    with open(meta_path, "w") as f:
        json.dump(make_json_safe(meta), f, indent=2)

    display(Markdown("# Microphone demo"))
    display(Markdown(f"**Model real-time factor:** `{result['model_rtf']:.3f}`  \\n**Model strength:** `{model_strength}`"))

    display(Markdown("### 1. Original mic input"))
    display(IPAudio(result["raw_16k"], rate=result["sample_rate"]))

    display(Markdown("### 2. Spectral-gate baseline (Essentially an algorithmic approach)"))
    display(IPAudio(result["baseline"], rate=result["sample_rate"]))

    display(Markdown("### 3. Model prediction"))
    display(IPAudio(result["model"], rate=result["sample_rate"]))

    zip_path = shutil.make_archive(str(run_dir), "zip", run_dir)
    print("Saved outputs:", run_dir.resolve())
    print("Saved zip:", Path(zip_path).resolve())
    return meta

print("Mic demo loaded. Run: mic_meta = mic_demo(seconds=8.0, model_strength=0.85, name='showcase_test')")

In [ ]:
# Run the live demo.
# Change seconds/model_strength if needed.

mic_meta = mic_demo(seconds=RECORD_SECONDS, model_strength=MODEL_STRENGTH, name="showcase_test")